<a href="https://colab.research.google.com/github/yusuf3662/softito/blob/main/tensorflow_and_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
SEED=42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

X,y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model =keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(8, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1,activation="sigmoid"),
])
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 641 (2.50 KB)

 Trainable params: 641 (2.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss=keras.losses.BinaryCrossentropy(),
              metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [ ]:
history = model.fit(X_train, y_train, validation_split=0.2, epochs=100, batch_size=4,verbose=2,callbacks=[early_stopping])

Epoch 1/100
91/91 - 1s - 15ms/step - accuracy: 0.3681 - loss: 0.9068 - val_accuracy: 0.3407 - val_loss: 0.8232
Epoch 2/100
91/91 - 0s - 3ms/step - accuracy: 0.3654 - loss: 0.8277 - val_accuracy: 0.3516 - val_loss: 0.7738
Epoch 3/100
91/91 - 0s - 4ms/step - accuracy: 0.3681 - loss: 0.8092 - val_accuracy: 0.3626 - val_loss: 0.7301
Epoch 4/100
91/91 - 0s - 5ms/step - accuracy: 0.3901 - loss: 0.7493 - val_accuracy: 0.3736 - val_loss: 0.6945
Epoch 5/100
91/91 - 1s - 7ms/step - accuracy: 0.3929 - loss: 0.7235 - val_accuracy: 0.4066 - val_loss: 0.6648
Epoch 6/100
91/91 - 1s - 7ms/step - accuracy: 0.4341 - loss: 0.6922 - val_accuracy: 0.4286 - val_loss: 0.6400
Epoch 7/100
91/91 - 1s - 7ms/step - accuracy: 0.4698 - loss: 0.6766 - val_accuracy: 0.4835 - val_loss: 0.6190
Epoch 8/100
91/91 - 0s - 3ms/step - accuracy: 0.4753 - loss: 0.6251 - val_accuracy: 0.4945 - val_loss: 0.6023
Epoch 9/100
91/91 - 0s - 3ms/step - accuracy: 0.5055 - loss: 0.6272 - val_accuracy: 0.5385 - val_loss: 0.5869
Epoch 10/

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader,random_split

In [ ]:
device =torch.device("cuda"if  torch.cuda.is_available() else "cpu")

In [ ]:
X_train_tensor = torch.tensor(X_train,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train,dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test,dtype=torch.float32).unsqueeze(1)

In [ ]:
full_ds = TensorDataset(X_train_tensor,y_train_tensor)
train_ds,val_ds = random_split(full_ds,[int(0.8*len(full_ds)),len(full_ds)-int(0.8*len(full_ds))])

train_loader = DataLoader(train_ds,batch_size=32,shuffle=True)
val_loader = DataLoader(val_ds,batch_size=32)

In [ ]:
class ANN(nn.Module):
  def __init__ (self,n_features):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_features,16),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(16,8),
        nn.ReLU(),
        nn.Dropout(0.2),
    )

  def forward(self,x):
    return self.net(x)

model = ANN(X_train.shape[1]).to(device)
print(model)

ANN(
  (net): Sequential(
    (0): Linear(in_features=30, out_features=16, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=16, out_features=8, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
  )
)


In [ ]:
criterion = nn.BCEWithLogitsLoss() # sigmoid + binary cross entropy
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
def accuracy(logits,targets):
  preds = (torch.sigmoid(logits)>0.5).float()
  return (preds==targets).float().mean()